# Laboratoire 2 — Intrication et inégalités de Bell

**Objectifs :**
- Générer et caractériser des états intriqués
- Calculer l'entropie d'intrication
- Violer les inégalités de Bell en simulation

**Bibliothèques :** QuTiP, Qiskit, Cirq

In [ ]:
import numpy as np
import qutip as qt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
%matplotlib inline

---
## 1. Fondement théorique

Les **états de Bell** sont les états maximalement intriqués à 2 qubits :

$$ \begin{aligned}
|\Phi^+\rangle &= \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle) \\
|\Phi^-\rangle &= \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle) \\
|\Psi^+\rangle &= \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle) \\
|\Psi^-\rangle &= \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)
\end{aligned} $$

Un état est **séparable** s'il peut s'écrire $\ket{\psi}_{AB} = \ket{\phi}_A \otimes \ket{\chi}_B$.
Sinon, il est **intriqué**.

---
## 2. États de Bell avec QuTiP

In [ ]:
ket00 = qt.tensor(qt.basis(2,0), qt.basis(2,0))
ket11 = qt.tensor(qt.basis(2,1), qt.basis(2,1))
ket01 = qt.tensor(qt.basis(2,0), qt.basis(2,1))
ket10 = qt.tensor(qt.basis(2,1), qt.basis(2,0))

phi_plus  = (ket00 + ket11).unit()
phi_minus = (ket00 - ket11).unit()
psi_plus  = (ket01 + ket10).unit()
psi_minus = (ket01 - ket10).unit()

print("|Φ⁺⟩ :")
print(phi_plus)
print("\n|Ψ⁻⟩ :")
print(psi_minus)

### Question 1
Vérifiez que ces 4 états forment une base orthonormée de $\mathbb{C}^4$ en calculant tous les produits scalaires.

In [ ]:
# Votre code ici
bell_states = [phi_plus, phi_minus, psi_plus, psi_minus]
names = ["|Φ⁺⟩", "|Φ⁻⟩", "|Ψ⁺⟩", "|Ψ⁻⟩"]

print("Matrice des produits scalaires :")
for i, si in enumerate(bell_states):
    row = []
    for j, sj in enumerate(bell_states):
        row.append(f"{si.dag() * sj:.2f}")
    print(f"{names[i]} : [" + ", ".join(row) + "]")

---
## 3. Matrice densité réduite et intrication

In [ ]:
rho_AB = phi_plus * phi_plus.dag()

# Trace partielle sur B
rho_A = rho_AB.ptrace(0)
rho_B = rho_AB.ptrace(1)

print("ρ_AB =")
print(rho_AB)
print("\nρ_A =")
print(rho_A)
print("\nρ_B =")
print(rho_B)

# Entropie d'intrication
print(f"\nS(ρ_A) = {qt.entropy_vn(rho_A):.4f}")
print(f"S(ρ_B) = {qt.entropy_vn(rho_B):.4f}")
print(f"Pour un état intriqué maximal : ln(2) = {np.log(2):.4f}")

### Question 2
Calculez l'entropie d'intrication pour les 4 états de Bell et pour un état séparable $\ket{00}$.
Que constatez-vous ?

In [ ]:
# Votre code ici
for name, state in zip(names, bell_states):
    rho = state * state.dag()
    E = qt.entropy_vn(rho.ptrace(0))
    print(f"E({name}) = {E:.4f}")

# État séparable
rho_sep = (ket00 * ket00.dag())
E_sep = qt.entropy_vn(rho_sep.ptrace(0))
print(f"E(|00⟩) = {E_sep:.4f}")

---
## 4. Intrication en fonction d'un paramètre

Considérons $\ket{\psi(\theta)} = \cos\theta\ket{00} + \sin\theta\ket{11}$.

In [ ]:
thetas = np.linspace(0, np.pi/2, 50)
entropies = []

for th in thetas:
    psi = np.cos(th) * ket00 + np.sin(th) * ket11
    psi = psi.unit()
    rho = psi * psi.dag()
    entropies.append(qt.entropy_vn(rho.ptrace(0)))

plt.plot(thetas, entropies)
plt.xlabel('θ (rad)')
plt.ylabel('S(ρ_A)')
plt.title('Entropie d\'intrication')
plt.axvline(np.pi/4, color='r', linestyle='--', label='θ=π/4 (max)')
plt.legend()
plt.grid()
plt.show()

---
## 5. Visualisation de la matrice densité

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (name, state) in enumerate(zip(names, bell_states)):
    qt.matrix_histogram(state * state.dag(), ax=axes[i])
    axes[i].set_title(name)
plt.tight_layout()
plt.show()

---
## 6. Test CHSH (inégalités de Bell)

Le jeu CHSH : Alice et Bob reçoivent $x, y \in \{0,1\}$ et doivent produire $a, b \in \{0,1\}$ tels que $a \oplus b = x \land y$.

- **Borne classique :** $S \leq 2$
- **Borne quantique :** $S = 2\sqrt{2} \approx 2.828$

In [ ]:
def mesure_CHSH(psi, theta_A, theta_B):
    """Valeur d'attente de cos(θ_A)Z + sin(θ_A)X ⊗ cos(θ_B)Z + sin(θ_B)X"""
    op_A = np.cos(theta_A) * qt.sigmaz() + np.sin(theta_A) * qt.sigmax()
    op_B = np.cos(theta_B) * qt.sigmaz() + np.sin(theta_B) * qt.sigmax()
    op = qt.tensor(op_A, op_B)
    return qt.expect(op, psi)

# Angles optimaux pour |Φ⁺⟩
angles = [
    (0, np.pi/4),
    (0, 3*np.pi/4),
    (np.pi/2, np.pi/4),
    (np.pi/2, 3*np.pi/4),
]

signes = [1, -1, 1, 1]  # S = E(a,b) - E(a,b') + E(a',b) + E(a',b')
S = sum(s * mesure_CHSH(phi_plus, thA, thB)
        for s, (thA, thB) in zip(signes, angles))

print(f"Valeur de S pour |Φ⁺⟩ : {S:.4f}")
print(f"Borne classique        : 2.0000")
print(f"Borne quantique max    : {2*np.sqrt(2):.4f}")
print(f"Violation ? {S > 2}")

### Question 3
Calculez $S$ pour les 4 états de Bell. Lequel donne la plus grande violation ?
Essayez avec l'état séparable $\ket{00}$. Que se passe-t-il ?

In [ ]:
# Votre code ici
for name, state in zip(names, bell_states):
    S = sum(s * mesure_CHSH(state, thA, thB)
            for s, (thA, thB) in zip(signes, angles))
    print(f"S({name}) = {S:.4f}, violation = {S > 2}")

# État séparable
S_sep = sum(s * mesure_CHSH(ket00, thA, thB)
            for s, (thA, thB) in zip(signes, angles))
print(f"S(|00⟩) = {S_sep:.4f}, violation = {S_sep > 2}")

---
## 7. Circuit Bell avec Qiskit

In [ ]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

print(qc.draw())

sim = AerSimulator()
result = sim.run(qc, shots=4096).result()
counts = result.get_counts()
print("\nRésultats :", counts)

# Les résultats sont parfaitement corrélés
correlated = sum(count for bits, count in counts.items() if bits[0] == bits[1])
print(f"Corrélés : {correlated/4096*100:.1f}%")
print(f"Non-corrélés : {(4096-correlated)/4096*100:.1f}%")

### Question 4
Modifiez le circuit pour produire $|\Psi^-\rangle$ au lieu de $|\Phi^+\rangle$.
Vérifiez les corrélations.

In [ ]:
# Votre code ici
qc_psi_minus = QuantumCircuit(2, 2)
qc_psi_minus.x(0)  # Commence par |10⟩
qc_psi_minus.h(0)
qc_psi_minus.cx(0, 1)
qc_psi_minus.measure([0, 1], [0, 1])

print(qc_psi_minus.draw())
result = sim.run(qc_psi_minus, shots=4096).result()
print("Résultats :", result.get_counts())

---
## 8. Implémentation Cirq

In [ ]:
import cirq

q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit([
    cirq.H(q0),
    cirq.CNOT(q0, q1),
    cirq.measure(q0, q1, key='result')
])

print("Circuit Cirq :")
print(circuit)

simulator = cirq.Simulator()
result = simulator.run(circuit, repetitions=4096)
print("\nRésultats Cirq :")
print(result.histogram(key='result'))

---
## 9. Exercices supplémentaires

1. Implémentez le test CHSH complet (avec mesures dans différentes bases) dans Qiskit.
2. Calculez la fidélité $F = \bra{\Phi^+}\rho\ket{\Phi^+}$ pour un état bruité par un canal dépolarisant.
3. Vérifiez le théorème de non-signalement : $\rho_A$ ne dépend pas des mesures sur $B$.
4. Générez un état de Werner $\rho = p|\Phi^+\rangle\langle\Phi^+| + (1-p)I/4$ et trouvez le seuil $p$ où il cesse d'être intriqué (critère PPT).

In [ ]:
# Exercice 2 : Fidélité sous bruit
from qutip import tensor, sigmax, sigmay, sigmaz, qeye

def depolarizing_channel(rho, p):
    K0 = np.sqrt(1 - 3*p/4) * tensor(qeye(2), qeye(2))
    K1 = np.sqrt(p/4) * tensor(sigmax(), qeye(2))
    K2 = np.sqrt(p/4) * tensor(sigmay(), qeye(2))
    K3 = np.sqrt(p/4) * tensor(sigmaz(), qeye(2))
    result = K0 * rho * K0.dag()
    for K in [K1, K2, K3]:
        result += K * rho * K.dag()
    return result

ps = np.linspace(0, 0.5, 20)
fidelities = []
rho_ideal = phi_plus * phi_plus.dag()

for p in ps:
    rho_noisy = depolarizing_channel(rho_ideal, p)
    F = (phi_plus.dag() * rho_noisy * phi_plus).real
    fidelities.append(F)

plt.plot(ps, fidelities)
plt.xlabel('p (bruit)')
plt.ylabel('Fidélité')
plt.grid()
plt.show()

In [ ]:
# Exercice 4 : État de Werner et seuil PPT
I4 = qt.tensor(qt.qeye(2), qt.qeye(2)) / 4  # État max mélangé sur 2 qubits

rhos = np.linspace(0, 1, 50)
entangled = []

for p in rhos:
    rho_W = p * phi_plus * phi_plus.dag() + (1-p) * I4
    rho_PT = qt.partial_transpose(rho_W, [0, 1])
    eigvals = rho_PT.eigenenergies()
    entangled.append(any(v < -1e-10 for v in eigvals))

plt.plot(rhos, entangled)
plt.xlabel('p')
plt.ylabel('Intriqué ?')
plt.axvline(1/3, color='r', linestyle='--', label='Seuil p=1/3')
plt.legend()
plt.grid()
plt.show()

print(f"Seuil théorique : p = 1/3 ≈ 0.333")
print(f"En dessous : état séparable (pas d'intrication)")
print(f"Au dessus : état intriqué")